In [1]:
import pandas as pd
import pymysql
from tqdm import tqdm
import os
import pickle

# DB 연결 함수

In [3]:
def connect_db():
    return pymysql.connect(
        host="127.0.0.1",
        user='root',
        password='240812',
        database='lol_data',
        port=3307
    )

# 챔피언 ID, 태그 사전 파일 저장 함수

In [17]:
def get_champion_dict(conn):
    champion_dict_query = "SELECT champion_id, champion_name, tags FROM champion_dict"
    champion_dict = pd.read_sql_query(champion_dict_query, conn)
    with open("table_data/champion_dict.pkl", "wb") as f:
        pickle.dump(champion_dict, f)
    return dict(zip(champion_dict['champion_id'], champion_dict['champion_name']))
conn = connect_db()
get_champion_dict(conn)
conn.close()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_6472\3920025925.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_dict = pd.read_sql_query(champion_dict_query, conn)


# 챔피언 참여자ID 라인타입 사전 파일 저장 함수

In [18]:
def get_champion_participant_id(conn):
    champion_participant_id_query = "SELECT match_id, participant_id, champion_id, position FROM champion_participant_id"
    champion_participant_id = pd.read_sql_query(champion_participant_id_query, conn)
    with open("table_data/champion_participant_id.pkl", "wb") as f:
        pickle.dump(champion_participant_id, f)
    return champion_participant_id
conn = connect_db()
get_champion_participant_id(conn)
conn.close()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_6472\3442806052.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_participant_id = pd.read_sql_query(champion_participant_id_query, conn)


# 게임 종료 파일 저장 함수

In [6]:
def get_game_end(conn):
    game_end_query = "SELECT match_id, real_timestamp, timestamp, winning_team FROM game_end"
    game_end = pd.read_sql_query(game_end_query, conn)
    with open("table_data/game_end.pkl", "wb") as f:
        pickle.dump(game_end, f)
    return game_end
conn = connect_db()
get_game_end(conn)
conn.close()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_6472\226839149.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  game_end = pd.read_sql_query(game_end_query, conn)


# 태그별 평균 수치 데이터 파일 저장함수

In [52]:
def tag_avg_stat():
    with open("dashboard_data/champion_stats.pkl", "rb") as f:
        champion_dataframes = pickle.load(f)

    with open("table_data/champion_dict.pkl", "rb") as f:
        champion_dict = pickle.load(f)

    # 챔피언 딕셔너리 데이터프레임 변환
    champion_df = pd.DataFrame(champion_dict)
    
    # 원하는 태그 매핑 이름 정의
    tag_mapping = {
        "Assassin": "암살자",
        "Fighter": "전사",
        "Mage": "마법사",
        "Marksman": "원거리 공격",
        "Support": "지원가",
        "Tank": "탱커"
    }

    # 결과를 저장할 딕셔너리
    tag_avg_dataframes = {}

    # 태그별 평균 계산
    for tag in set(tag for tags in champion_df['tags'] for tag in eval(tags)):

        # 해당 태그를 가진 챔피언 목록 필터링
        champions_with_tag = champion_df[champion_df['tags'].apply(lambda x: tag in eval(x))]['champion_name'].tolist()
        
        # 태그에 해당하는 챔피언들의 데이터 합산 및 평균 계산
        tag_data = pd.concat([champion_dataframes[champion] for champion in champions_with_tag if champion in champion_dataframes])
        
        # 각 분(minute)마다의 평균을 구함
        tag_avg_df = tag_data.groupby("분").mean().reset_index()


        mapped_tag = tag_mapping.get(tag, tag)  # tag_mapping에 없으면 원래 태그 이름 사용
        tag_avg_dataframes[mapped_tag] = tag_avg_df

    # 최종 태그별 평균 데이터 저장
    with open("dashboard_data/tag_avg_stats.pkl", "wb") as f:
        pickle.dump(tag_avg_dataframes, f)

    print("Tag-based average stats saved to tag_avg_stats.pkl")
tag_avg_stat()

Tag-based average stats saved to tag_avg_stats.pkl


In [55]:
with open("dashboard_data/tag_avg_stats.pkl", "rb") as f:
    data = pickle.load(f)
data['암살자']

,분,챔피언에게 가한 피해량,받은 피해량,총 획득한 골드량,총 경험치량,평균 누적 와드 설치 횟수,평균 누적 와드 제거 횟수
0,0,0.000000,0.000000,500.000000,0.000000,1.001384,1.001105
1,1,37.190909,40.277273,509.090909,0.381818,1.012552,1.022814
2,2,206.363636,452.188636,613.456818,216.065909,1.057364,1.091973
3,3,513.815909,1109.261364,979.215909,741.227273,1.328016,1.064061
4,4,883.986364,1699.111364,1352.297727,1171.547727,2.003402,1.122814
5,5,1267.227273,2299.868182,1724.281818,1616.665909,2.285352,1.150950
6,6,1651.347727,2939.709091,2093.354545,2067.054545,2.660780,1.200677
7,7,2095.388636,3671.975000,2462.718182,2491.743182,3.346120,1.306948
8,8,2562.150000,4425.843182,2863.747727,2971.970455,3.716641,1.438280
9,9,3068.165909,5217.515909,3270.765909,3451.877273,4.208959,1.579148


# 이벤트 파일 합치는 함수

In [45]:
def load_and_merge_champion_data():
    # 챔피언 데이터 로드
    with open("dashboard_data/champion_main_stats.pkl", "rb") as f:
        champion_main_stats = pickle.load(f)
    
    with open("dashboard_data/champion_ward_placement_stats.pkl", "rb") as f:
        champion_ward_placement_stats = pickle.load(f)
    
    with open("dashboard_data/champion_ward_removal_stats.pkl", "rb") as f:
        champion_ward_removal_stats = pickle.load(f)
    
    merged_data = {}

    for champion_name in champion_main_stats.keys():
        main_stats = champion_main_stats.get(champion_name)
        placement_stats = champion_ward_placement_stats.get(champion_name)
        removal_stats = champion_ward_removal_stats.get(champion_name)
        
        # 분 기준으로 병합
        merged_df = pd.merge(main_stats, placement_stats, on='분', how='outer')
        merged_df = pd.merge(merged_df, removal_stats, on='분', how='outer')
        
        # 분 기준으로 정렬하고 중간 NaN 방지
        merged_df = merged_df.sort_values('분').reset_index(drop=True)
        
        # 병합된 데이터프레임을 딕셔너리에 저장
        merged_data[champion_name] = merged_df
    
    with open("dashboard_data/champion_stats.pkl", "wb") as f:
        pickle.dump(merged_data, f)

In [46]:
load_and_merge_champion_data()

In [47]:
with open("dashboard_data/champion_stats.pkl", "rb") as f:
    data = pickle.load(f)
data

{'애니':      분  챔피언에게 가한 피해량   받은 피해량  총 획득한 골드량   총 경험치량  평균 누적 와드 설치 횟수  \
 0    0           0.0      0.0      500.0      0.0          1.0000   
 1    1          51.1     38.8      515.8      0.7          1.0000   
 2    2         270.4    239.2      629.0    218.4          1.0185   
 3    3         616.7    574.5      962.6    805.3          1.2028   
 4    4         939.6    917.2     1279.3   1294.2          1.9385   
 5    5        1237.7   1274.8     1618.8   1828.5          2.0708   
 6    6        1625.4   1673.7     1957.4   2341.2          2.3812   
 7    7        2152.7   2084.7     2313.8   2814.0          3.1545   
 8    8        2647.7   2525.0     2692.2   3329.7          3.5296   
 9    9        3221.3   3003.7     3085.4   3841.4          4.1809   
 10  10        3751.6   3456.1     3439.8   4309.7          4.8265   
 11  11        4356.3   3983.3     3858.2   4845.2          5.1546   
 12  12        5051.5   4579.7     4283.4   5344.8          5.8474   
 13  13       

# 챔피언 기본 정보 저장 함수

In [ ]:
# 챔피언 기본 스탯 추출 쿼리
def get_champion_main_stats(conn):
    main_stats_query = """
    SELECT 
        cp.champion_id,
        FLOOR(cs.timestamp / 60000) AS minute,
        ROUND(SUM(cs.tdd_to_champion), 1) AS total_tdd_to_champion,
        ROUND(SUM(cs.total_damage_taken), 1) AS total_damage_taken,
        ROUND(SUM(cs.total_gold), 1) AS total_gold,
        ROUND(SUM(cs.xp), 1) AS total_xp
    FROM 
        champion_stat_per_timestamp cs
    JOIN 
        champion_participant_id cp ON cs.match_id = cp.match_id AND cs.participant_id = cp.participant_id
    GROUP BY 
        cp.champion_id, FLOOR(cs.timestamp / 60000)
    ORDER BY 
        cp.champion_id, minute;
    """
    return pd.read_sql_query(main_stats_query, conn)

In [ ]:
def save_main_stats_to_pkl():
    conn = connect_db()
    champion_dict = get_champion_dict(conn)
    main_stats = get_champion_main_stats(conn)
    conn.close()

    column_mapping = {
        "minute": "분",
        "avg_tdd_to_champion": "챔피언에게 가한 피해량",
        "avg_total_damage_taken": "받은 피해량",
        "avg_total_gold": "총 획득한 골드량",
        "avg_xp": "총 경험치량",
    }
    champion_dataframes = {}
    for champion_id, champion_name in tqdm(champion_dict.items(), desc="Processing Champions"):
        champion_df = main_stats[main_stats['champion_id'] == champion_id].drop(columns=['champion_id'])
        champion_df = champion_df.rename(columns=column_mapping)
        champion_dataframes[champion_name] = champion_df
    with open("dashboard_data/champion_main_stats.pkl", "wb") as f:
        pickle.dump(champion_dataframes, f)

In [ ]:
with open("data/champion_main_stats.pkl", "rb") as f:
    champion_main_stats_data = pickle.load(f)
champion_main_stats_data['가렌']

{'애니':      분  챔피언에게 가한 피해량   받은 피해량  총 획득한 골드량   총 경험치량
 0    0           0.0      0.0      500.0      0.0
 1    1          51.1     38.8      515.8      0.7
 2    2         270.4    239.2      629.0    218.4
 3    3         616.7    574.5      962.6    805.3
 4    4         939.6    917.2     1279.3   1294.2
 5    5        1237.7   1274.8     1618.8   1828.5
 6    6        1625.4   1673.7     1957.4   2341.2
 7    7        2152.7   2084.7     2313.8   2814.0
 8    8        2647.7   2525.0     2692.2   3329.7
 9    9        3221.3   3003.7     3085.4   3841.4
 10  10        3751.6   3456.1     3439.8   4309.7
 11  11        4356.3   3983.3     3858.2   4845.2
 12  12        5051.5   4579.7     4283.4   5344.8
 13  13        5748.2   5174.8     4661.9   5768.5
 14  14        6489.0   5786.7     5086.3   6262.6
 15  15        7318.2   6399.1     5538.5   6744.0
 16  16        8179.9   7039.1     5918.6   7223.9
 17  17        9118.5   7711.3     6316.8   7695.9
 18  18       10078.8   8

# 와드

In [ ]:
# 와드 제거 횟수 데이터 추출
def get_ward_removal_stats(conn):
    ward_removal_query = """
    WITH filtered_ward_kills AS (
        SELECT 
            wk.match_id,
            cp.champion_id,
            FLOOR(wk.timestamp / 60000) AS minute,
            COUNT(*) AS ward_removals
        FROM 
            ward_kill wk
        JOIN 
            champion_participant_id cp ON wk.match_id = cp.match_id AND wk.participant_id = cp.participant_id
        WHERE 
            wk.ward_type != 'UNDEFINED'
        GROUP BY 
            wk.match_id, cp.champion_id, minute
    ),
    cumulative_ward_removals AS (
        SELECT 
            match_id,
            champion_id,
            minute,
            SUM(ward_removals) OVER (PARTITION BY match_id, champion_id ORDER BY minute ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_removals
        FROM 
            filtered_ward_kills
    )
    SELECT 
        champion_id,
        minute,
        AVG(cumulative_removals) AS avg_cumulative_ward_removals
    FROM 
        cumulative_ward_removals
    GROUP BY 
        champion_id, minute
    ORDER BY 
        champion_id, minute;
    """
    return pd.read_sql_query(ward_removal_query, conn)

# 와드 설치 횟수 데이터 추출
def get_ward_placement_stats(conn):
    ward_placement_query = """
    WITH filtered_ward_placements AS (
        SELECT 
            wp.match_id,
            cp.champion_id,
            FLOOR(wp.timestamp / 60000) AS minute,
            COUNT(*) AS ward_placements
        FROM 
            ward_placed wp
        JOIN 
            champion_participant_id cp ON wp.match_id = cp.match_id AND wp.participant_id = cp.participant_id
        GROUP BY 
            wp.match_id, cp.champion_id, minute
    ),
    cumulative_ward_placements AS (
        SELECT 
            match_id,
            champion_id,
            minute,
            SUM(ward_placements) OVER (PARTITION BY match_id, champion_id ORDER BY minute ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_placements
        FROM 
            filtered_ward_placements
    )
    SELECT 
        champion_id,
        minute,
        AVG(cumulative_placements) AS avg_cumulative_ward_placements
    FROM 
        cumulative_ward_placements
    GROUP BY 
        champion_id, minute
    ORDER BY 
        champion_id, minute;
    """
    return pd.read_sql_query(ward_placement_query, conn)

def save_ward_removal_stats_to_pkl():
    conn = connect_db()
    ward_removal_stats = get_ward_removal_stats(conn)
    ward_placement_stats = get_ward_placement_stats(conn)
    conn.close()  
    with open("table_data/ward_removal_stats.pkl", "wb") as f:
        pickle.dump(ward_removal_stats, f)
    with open("table_data/ward_placement_stats.pkl", "wb") as f:
        pickle.dump(ward_placement_stats, f)

C:\Users\dgjja\AppData\Local\Temp\ipykernel_7004\3701261899.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(ward_removal_query, conn)
C:\Users\dgjja\AppData\Local\Temp\ipykernel_7004\3701261899.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(ward_placement_query, conn)


In [ ]:
save_ward_removal_stats_to_pkl()

## 챔피언별 와드제거 데이터 저장

In [33]:
def save_champion_ward_removal_stats_to_pkl():
    conn = connect_db()
    champion_dict = get_champion_dict(conn)
    conn.close()  
    with open("table_data/ward_removal_stats.pkl", "rb") as f:
        ward_removal_stats = pickle.load(f)
    column_mapping = {
        "minute": "분",
        "avg_cumulative_ward_removals": "평균 누적 와드 제거 횟수",
    }
    champion_dataframes = {}
    for champion_id, champion_name in tqdm(champion_dict.items(), desc="Processing Champions"):
        removal_df = ward_removal_stats[ward_removal_stats['champion_id'] == champion_id][['minute', 'avg_cumulative_ward_removals']]
        removal_df = removal_df.rename(columns=column_mapping)
        champion_dataframes[champion_name] = removal_df
    with open("dashboard_data/champion_ward_removal_stats.pkl", "wb") as f:
        pickle.dump(champion_dataframes, f)

In [34]:
save_champion_ward_removal_stats_to_pkl()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_7004\3920025925.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_dict = pd.read_sql_query(champion_dict_query, conn)
Processing Champions: 100%|██████████| 168/168 [00:00<00:00, 962.57it/s]


## 챔피언별 와드 설치 데이터 저장

In [35]:
def save_champion_ward_placement_stats_to_pkl():
    conn = connect_db()
    champion_dict = get_champion_dict(conn)
    conn.close()  
    with open("table_data/ward_placement_stats.pkl", "rb") as f:
        ward_placement_stats = pickle.load(f)
    column_mapping = {
        "minute": "분",
        "avg_cumulative_ward_placements": "평균 누적 와드 설치 횟수",
    }
    champion_dataframes = {}
    for champion_id, champion_name in tqdm(champion_dict.items(), desc="Processing Champions"):
        placement_df = ward_placement_stats[ward_placement_stats['champion_id'] == champion_id][['minute', 'avg_cumulative_ward_placements']]
        placement_df = placement_df.rename(columns=column_mapping)
        champion_dataframes[champion_name] = placement_df
    with open("dashboard_data/champion_ward_placement_stats.pkl", "wb") as f:
        pickle.dump(champion_dataframes, f)

In [37]:
save_champion_ward_placement_stats_to_pkl()

C:\Users\dgjja\AppData\Local\Temp\ipykernel_7004\3920025925.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  champion_dict = pd.read_sql_query(champion_dict_query, conn)
Processing Champions: 100%|██████████| 168/168 [00:00<00:00, 1027.02it/s]


In [39]:
with open('dashboard_data/champion_main_stats.pkl','rb') as f:
    data = pickle.load(f)
data['닐라'].head(5)

,분,챔피언에게 가한 피해량,받은 피해량,총 획득한 골드량,총 경험치량
8116,0,0.0,0.0,500.0,0.0
8117,1,34.2,35.2,509.9,0.4
8118,2,200.7,291.4,601.6,116.0
8119,3,444.2,726.2,962.4,577.5
8120,4,783.0,1158.6,1360.6,988.3


# 챔피언 승률

In [58]:
def get_champion_win_pick_rate():
    with open('table_data/champion_participant_id.pkl', 'rb') as f:
        champion_participant_id = pickle.load(f)
    with open('table_data/game_end.pkl', 'rb') as f:
        game_end = pickle.load(f)
    with open('table_data/champion_dict.pkl', 'rb') as f:
        champion_dict = pickle.load(f)

    # 데이터 복사
    champion_participant_df = champion_participant_id.copy()
    game_end_df = game_end.copy()
    champion_dict_df = champion_dict[['champion_id', 'champion_name']]

    # 팀 정보 추가
    champion_participant_df['team'] = champion_participant_df['participant_id'].apply(lambda x: 100 if x <= 5 else 200)

    # game_end_df와 병합하여 각 챔피언이 승리했는지 여부 계산
    merged_df = champion_participant_df.merge(game_end_df[['match_id', 'winning_team']], on='match_id')
    merged_df['win'] = merged_df['team'] == merged_df['winning_team']

    # 각 position, champion_id별로 승리 횟수, 총 픽 횟수 집계
    grouped_df = merged_df.groupby(['position', 'champion_id']).agg(
        total_picks=('match_id', 'count'),
        wins=('win', 'sum')
    ).reset_index()

    # 포지션별 경기 수 집계
    position_match_counts = merged_df.groupby('position')['match_id'].nunique().reset_index(name='position_match_counts')
    grouped_df = grouped_df.merge(position_match_counts, on='position')

    # 승률 및 픽률 계산
    grouped_df['win_rate'] = (grouped_df['wins'] / grouped_df['total_picks']).round(4)
    grouped_df['pick_rate'] = (grouped_df['total_picks'] / grouped_df['position_match_counts']).round(4)

    # 챔피언 이름을 추가하기 위해 champion_dict와 병합
    grouped_df = grouped_df.merge(champion_dict_df, on='champion_id')

    # 최종 결과물 구성, 컬럼 이름 수정 및 필요 없는 컬럼 제거
    result_df = grouped_df[['champion_name', 'win_rate', 'pick_rate', 'total_picks']]
    result_df.columns = ['챔피언', '승률', '픽률', '표본 수']

    # 포지션별로 데이터프레임을 분리하여 딕셔너리에 저장
    position_dict = {position: df for position, df in result_df.groupby(grouped_df['position'])}

    with open("dashboard_data/champion_win_pick_rate.pkl", "wb") as f:
        pickle.dump(position_dict, f)

In [59]:
get_champion_win_pick_rate()

In [60]:
with open('dashboard_data/champion_win_pick_rate.pkl', 'rb') as f:
    champion_dict = pickle.load(f)

In [64]:
champion_dict['BOTTOM'].sort_values(by='픽률', ascending=False)

,챔피언,승률,픽률,표본 수
89,진,0.4923,0.2865,17140
16,애쉬,0.5130,0.2324,13902
79,카이사,0.4918,0.2261,13526
27,코르키,0.5270,0.1323,7912
92,징크스,0.5109,0.1298,7768
...,...,...,...,...
28,카르마,0.5000,0.0000,2
26,갱플랭크,0.0000,0.0000,1
86,카밀,1.0000,0.0000,1
88,브라움,0.0000,0.0000,1


In [41]:
position_dict['BOTTOM']['pick_rate'].sum()

2.0000668605622973

In [42]:
position_dict['MIDDLE'].sort_values(by='pick_rate', ascending=False)

,champion_id,position,win_rate,pick_rate
369,517,MIDDLE,0.509899,0.186591
374,777,MIDDLE,0.486168,0.147428
283,61,MIDDLE,0.497893,0.111089
269,42,MIDDLE,0.510802,0.097483
312,103,MIDDLE,0.497837,0.096597
...,...,...,...,...
256,28,MIDDLE,0.000000,0.000017
379,895,MIDDLE,1.000000,0.000017
364,421,MIDDLE,0.000000,0.000017
348,222,MIDDLE,1.000000,0.000017


In [33]:
position_dict['TOP'].sort_values(by='pick_rate', ascending=False)

,champion_id,position,win_rate,pick_rate
542,897,TOP,0.494397,0.180473
407,24,TOP,0.515368,0.138134
497,150,TOP,0.505318,0.110019
540,893,TOP,0.522938,0.079430
437,58,TOP,0.494810,0.077291
...,...,...,...,...
534,555,TOP,1.000000,0.000017
432,53,TOP,1.000000,0.000017
411,28,TOP,1.000000,0.000017
461,89,TOP,0.000000,0.000017


In [35]:
position_dict['TOP']['pick_rate'].sum()

2.000083575702871